# Fib (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_java.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Fib`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fibonacci numbers: Fib

**Input:** a natural number $n$. **Output:** $F(n)$, with $F(0) = 0$, $F(1) = 1$ and $F(n) = F(n-1) + F(n-2)$. Naive recursive algorithm: $C(n) = 2F(n+1) - 1$ calls and $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ steps.

Pseudo-code: section 14.5 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile Fib.java
public class Fib
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Fib(n)
    static long fib(int n)
    {
        push(n); cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            if (trace) say(String.format("Fib(%d)", n), String.format("base case, returns %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("Fib(%d)", n), String.format("calls Fib(%d)", n - 1));
        long x = fib(n - 1);                       // the two calls are separated only to print the trace
        if (trace) say(String.format("Fib(%d)", n), String.format("calls Fib(%d)", n - 2));
        long y = fib(n - 2);
        cnt[5]++;
        if (trace) say(String.format("Fib(%d)", n), String.format("returns %d + %d = %d", x, y, x + y));
        pop();
        return x + y;
    }

    static long fibValue(int k)                    // F(k) with a loop, to check
    {
        long a = 0, b = 1;
        for (int i = 0; i < k; i++)
        {
            long t = a + b;
            a = b;
            b = t;
        }
        return a;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: the execution of Fib(4), in depth-first order
        int n = 4;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        long r = fib(n);
        trace = false;
        System.out.printf("F(%d) = %d, calls C(%d) = %d = 2F(%d) - 1, depth = %d\n", n, r, n, calls, n + 1, maxDepth);

        // Fib(5): the same subproblems are solved many times
        System.out.printf("\n");
        n = 5;
        reset();
        r = fib(n);
        System.out.printf("Fib(%d) = %d: %d calls, depth %d, S = %d steps\n", n, r, calls, maxDepth, total());
        System.out.printf("  k | calls of Fib(k)\n");
        for (int k = n; k >= 0; k--) System.out.printf("%3d | %d\n", k, seen[k]);
        printSteps(2, 3, 5);

        // Number of calls and steps: C(n) = 2F(n+1) - 1 and S(n) = 4F(n+1) - 2
        System.out.printf("\n");
        System.out.printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | depth\n");
        int[] values = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = fib(v);
            System.out.printf("%3d | %6d | %8d | %11d | %8d | %11d | %d\n", v, r, calls, 2 * fibValue(v + 1) - 1, total(), 4 * fibValue(v + 1) - 2, maxDepth);
        }
    }
}

Writing Fib.java


In [3]:
!javac Fib.java && java Fib

call                        | action
Fib(4)                      | calls Fib(3)
  Fib(3)                    | calls Fib(2)
    Fib(2)                  | calls Fib(1)
      Fib(1)                | base case, returns 1
    Fib(2)                  | calls Fib(0)
      Fib(0)                | base case, returns 0
    Fib(2)                  | returns 1 + 0 = 1
  Fib(3)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(3)                    | returns 1 + 1 = 2
Fib(4)                      | calls Fib(2)
  Fib(2)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(2)                    | calls Fib(0)
    Fib(0)                  | base case, returns 0
  Fib(2)                    | returns 1 + 0 = 1
Fib(4)                      | returns 2 + 1 = 3
F(4) = 3, calls C(4) = 9 = 2F(5) - 1, depth = 4

Fib(5) = 5: 15 calls, depth 5, S = 30 steps
  k | calls of Fib(k)
  5 | 1
  4 | 1
  3 | 2
  2 | 3
  1 | 5
  0 | 3
Line | Ex

**Try it yourself.** Change $n$ and see how fast the calls grow.

In [4]:
%%writefile ProvaFib.java
public class ProvaFib
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- change n (with n > 45 it takes a while)
        Fib.reset();
        long r = Fib.fib(n);
        System.out.printf("F(%d) = %d, calls = %d, S = %d\n", n, r, Fib.calls, Fib.total());
    }
}

Writing ProvaFib.java


In [5]:
!javac ProvaFib.java && java ProvaFib

F(6) = 8, calls = 25, S = 50


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).